In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Raiz do projeto = pasta acima de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

print("chave definida:", bool(os.environ.get("USGS_API_KEY")))


chave definida: False


## 1. Configuração

In [11]:
import os, time, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt


pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_rows = 80
plt.rcParams["figure.dpi"] = 110

# ----------------- PARÂMETROS -----------------
API_VERSION = "v0"                 # "v0" ou "v1": a USGS mantém as duas por um período. Veja a seção 2.
BASE = f"https://api.waterdata.usgs.gov/ogcapi/{API_VERSION}"
API_KEY = os.environ.get("USGS_API_KEY")

DATE_START, DATE_END = "2010-01-01", "2025-12-31"
PARAMETER_CODE = "00060"           # vazão (discharge), ft3/s
STATISTIC_ID = "00003"             # média diária

STATES = None                      # None = todos os estados do catálogo; ou ex.: ["Ohio", "Texas"]
REQUIRE_FULL_COVERAGE = False       # só séries cujo período de registro cobre 2010-2024 inteiro
ALL_COLUMNS = True                 # True = pede TODAS as colunas que a API oferece (sem filtro "properties")
COVARIATES = None                 # "all", lista de códigos (ex.: ["00065", "00010"]) ou None
COVARIATE_STATS = ["00003"]        # estatísticas diárias aceitas nas covariáveis (média). None = todas
MAX_SITES = None                    #usa todos os estados do catálogo.
# O catálogo precisa estar completo para o código da seção 4 conseguir descobrir qual estado tem mais estações.
#A restrição a um estado acontece lá, não aqui.
SEED = 42
BATCH_SIZE = 8                     # séries por requisição (8 x 5479 dias < limite de 50.000 linhas)
MIN_COMPLETENESS = 0.95            # fração mínima de dias presentes para entrar no dataset final

CFS_TO_M3S = 0.028316846592
EXPECTED_DAYS = len(pd.date_range(DATE_START, DATE_END, freq="D"))

DATA_DIR = PROJECT_ROOT / "data" / "usgs"
RAW_DIR, PROC_DIR = DATA_DIR / "raw", DATA_DIR / "raw" / "parquets"
for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not API_KEY:
    print("AVISO: USGS_API_KEY não definida. Sem chave o limite por hora é baixo (erro 429 é provável).")
print(f"Período: {DATE_START} a {DATE_END} ({EXPECTED_DAYS} dias esperados por estação)")

AVISO: USGS_API_KEY não definida. Sem chave o limite por hora é baixo (erro 429 é provável).
Período: 2010-01-01 a 2025-12-31 (5844 dias esperados por estação)


In [12]:
session = requests.Session()
if API_KEY:
    session.headers["X-Api-Key"] = API_KEY

def get_json(url, params=None, retries=6):
    for attempt in range(retries):
        r = session.get(url, params=params, timeout=180)
        if r.status_code == 429:
            try:
                wait = float(r.headers.get("Retry-After", ""))
            except ValueError:
                wait = 15 * 2 ** attempt
            print(f"  429 (limite de taxa); aguardando {wait:.0f}s ...")
            time.sleep(min(wait, 3700)); continue
        if r.status_code in (500, 502, 503, 504):
            time.sleep(3 * 2 ** attempt); continue
        if r.status_code == 403:
            raise RuntimeError("403 Forbidden: requisição grande demais ou bloqueada. Reduza BATCH_SIZE.")
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Falha após {retries} tentativas: {url}")

def fetch_items(collection, params, limit=50000):
    """Percorre todas as páginas (link rel=next) e devolve DataFrame com as propriedades,
    o id da feição e lon/lat quando houver geometria de ponto."""
    url = f"{BASE}/collections/{collection}/items"
    p = {"f": "json", "limit": limit, **params}
    rows, first = [], True
    while url:
        js = get_json(url, p if first else None)
        first = False
        for ft in js.get("features", []):
            rec = dict(ft.get("properties") or {})
            rec["_feature_id"] = ft.get("id")
            g = ft.get("geometry")
            if g and g.get("type") == "Point":
                rec["lon"], rec["lat"] = g["coordinates"][:2]
            rows.append(rec)
        url = next((l["href"] for l in js.get("links", []) if l.get("rel") == "next"), None)
    if len(rows) >= 50000:
        print(f"  AVISO: {len(rows):,} linhas em '{collection}'. Pode ter batido o teto de 50.000 por requisição.")
    return pd.DataFrame(rows)

## 2. Verificação do esquema da API

A USGS tem duas versões da API ao mesmo tempo (`v0` e `v1`), e o nome de alguns campos mudou entre elas (por exemplo, o período de registro era `begin`/`end` e passou a ter `begin_utc`/`end_utc`; a migração cita `start`/`end`). Algumas páginas da documentação ainda mostram nomes antigos. Em vez de supor, o notebook lê o esquema real e escolhe os campos.

In [13]:
def get_schema_props(collection):
    return list(get_json(f"{BASE}/collections/{collection}/schema", {"f": "json"}).get("properties", {}))

ts_props = get_schema_props("time-series-metadata")
daily_props = get_schema_props("daily")
print("time-series-metadata:", ts_props)
print("daily:", daily_props)

def pick(options, props, what):
    for o in options:
        if o in props:
            return o
    raise RuntimeError(f"Nenhum campo de {what} encontrado entre {options} em {props}. "
                       f"Tente API_VERSION='v1' (ou 'v0') e revise a documentação.")

BEGIN_FIELD = pick(["begin_utc", "start", "begin"], ts_props, "início do período de registro")
END_FIELD = pick(["end_utc", "end"], ts_props, "fim do período de registro")
for need in ["state_name", "monitoring_location_id", "parameter_code", "statistic_id"]:
    if need not in ts_props:
        raise RuntimeError(f"Campo '{need}' ausente em time-series-metadata nesta versão da API.")
for need in ["time_series_id", "time", "value"]:
    if need not in daily_props:
        raise RuntimeError(f"Campo '{need}' ausente em daily nesta versão da API.")
print(f"\nPeríodo de registro: {BEGIN_FIELD} / {END_FIELD}")

time-series-metadata: ['geometry', 'id', 'unit_of_measure', 'parameter_name', 'parameter_code', 'statistic_id', 'hydrologic_unit_code', 'state_name', 'last_modified', 'begin', 'end', 'begin_utc', 'end_utc', 'computation_period_identifier', 'computation_identifier', 'thresholds', 'sublocation_identifier', 'primary', 'monitoring_location_id', 'web_description', 'parameter_description', 'parent_time_series_id', 'data_gap_interval']
daily: ['geometry', 'time_series_id', 'monitoring_location_id', 'parameter_code', 'statistic_id', 'time', 'value', 'unit_of_measure', 'approval_status', 'qualifier', 'last_modified']

Período de registro: begin_utc / end_utc


## 2.1 Explorar: colunas e estados disponíveis

Rode estas duas células quando quiser **ver** o que a API oferece, sem baixar dados. A primeira lista as coleções e todas as colunas de cada uma (nome, título, tipo, descrição). A segunda lista todos os estados/territórios que têm vazão diária, com o número de estações.

Com `ALL_COLUMNS = True` (padrão), o notebook baixa **todas** essas colunas. A documentação da USGS recomenda limitar as colunas para o download ficar mais rápido, então se ficar lento ou pesado, use `ALL_COLUMNS = False`.

In [14]:
colecoes = [c["id"] for c in get_json(f"{BASE}/collections", {"f": "json"}).get("collections", [])]
print("Coleções disponíveis nesta versão da API:", colecoes)

def schema_table(collection):
    props = get_json(f"{BASE}/collections/{collection}/schema", {"f": "json"}).get("properties", {})
    return pd.DataFrame([{"coluna": k, "titulo": v.get("title"),
                          "tipo": v.get("type") or v.get("format"),
                          "descricao": (v.get("description") or "").split("\n")[0][:110]}
                         for k, v in props.items()])

for coll in ["monitoring-locations", "time-series-metadata", "daily"]:
    t = schema_table(coll)
    print(f"\n=== {coll}: {len(t)} colunas ===")
    display(t)

Coleções disponíveis nesta versão da API: ['latest-continuous', 'continuous', 'latest-daily', 'daily', 'edr/daily', 'latest-field-measurements', 'field-measurements', 'channel-measurements', 'combined-metadata', 'field-measurements-metadata', 'time-series-metadata', 'monitoring-locations', 'peaks', 'time-series-revisions', 'time-series-methods', 'agency-codes', 'altitude-datums', 'aquifer-codes', 'aquifer-types', 'coordinate-accuracy-codes', 'coordinate-datum-codes', 'coordinate-method-codes', 'counties', 'countries', 'hydrologic-unit-codes', 'medium-codes', 'methods', 'method-categories', 'method-citations', 'citations', 'combined-method-citations', 'national-aquifer-codes', 'parameter-codes', 'reliability-codes', 'site-types', 'states', 'statistic-codes', 'topographic-codes', 'time-zone-codes']

=== monitoring-locations: 44 colunas ===


,coluna,titulo,tipo,descricao
0,geometry,NaN,geometry-any,
1,id,Monitoring location ID,string,A unique identifier representing a single moni...
2,agency_code,Agency code,string,The agency that is reporting the data. Agency ...
3,agency_name,Agency name,string,The name of the agency that is reporting the d...
4,monitoring_location_number,Monitoring location number,string,Each monitoring location in the USGS data base...
5,monitoring_location_name,Monitoring location name,string,This is the official name of the monitoring lo...
6,district_code,District code,string,The Water Science Centers (WSCs) across the Un...
7,country_code,Country code,string,The code for the country in which the monitori...
8,country_name,Country name,string,The name of the country in which the monitorin...
9,state_code,State code,string,State code. A [two-digit ANSI code](https://ww...



=== time-series-metadata: 23 colunas ===


,coluna,titulo,tipo,descricao
0,geometry,NaN,geometry-any,
1,id,Time series ID,string,A unique identifier representing a single time...
2,unit_of_measure,Unit of measure,string,A human-readable description of the units of m...
3,parameter_name,Parameter name,string,A human-understandable name corresponding to `...
4,parameter_code,Parameter code,string,Parameter codes are 5-digit codes used to iden...
5,statistic_id,Statistic ID,string,A code corresponding to the statistic an obser...
6,hydrologic_unit_code,Hydrologic unit code (HUC),string,The hydrologic unit code (HUC) for the monitor...
7,state_name,State name,string,The name of the state or state equivalent in w...
8,last_modified,Last modified date,string,The last time a record was refreshed in our da...
9,begin,Time series first observation timestamp - loca...,string,"This field contains the same information as ""b..."



=== daily: 11 colunas ===


,coluna,titulo,tipo,descricao
0,geometry,NaN,geometry-any,
1,time_series_id,Time series ID,string,A unique identifier representing a single time...
2,monitoring_location_id,Monitoring location ID,string,A unique identifier representing a single moni...
3,parameter_code,Parameter code,string,Parameter codes are 5-digit codes used to iden...
4,statistic_id,Statistic ID,string,A code corresponding to the statistic an obser...
5,time,Time,string,The date an observation represents. You can qu...
6,value,Value,string,The value of the observation. Values are trans...
7,unit_of_measure,Unit of measure,string,A human-readable description of the units of m...
8,approval_status,Approval Status,string,Some of the data that you have obtained from t...
9,qualifier,Qualifier,string,This field indicates any qualifiers associated...


In [15]:
p = {"parameter_code": PARAMETER_CODE, "statistic_id": STATISTIC_ID,
     "skipGeometry": "true", "properties": "state_name,monitoring_location_id"}
if "computation_period_identifier" in ts_props:
    p["computation_period_identifier"] = "Daily"

tmp = fetch_items("time-series-metadata", p)
estados = (tmp.groupby("state_name")["monitoring_location_id"].nunique()
              .sort_values(ascending=False).rename("estacoes_com_vazao_diaria"))
print(f"{len(estados)} estados/territórios com vazão média diária:")
display(estados.to_frame())

66 estados/territórios com vazão média diária:


,estacoes_com_vazao_diaria
state_name,
California,2429
Colorado,1293
Washington,1064
Florida,1027
Texas,948
Idaho,904
Montana,903
Wisconsin,787
Utah,777


In [ ]:
# salva o resultado da consulta acima (tmp e estados) em parquet
tmp_path = RAW_DIR / "states_timeseries_metadata.parquet"
estados_path = RAW_DIR / "states_daily_discharge_counts.parquet"
tmp.to_parquet(tmp_path)
estados.to_frame().to_parquet(estados_path)
print(f"Salvo: {tmp_path}")
print(f"Salvo: {estados_path}")

## 3. Catálogo: estados e estações disponíveis

Duas consultas:
1. `time-series-metadata`: todas as séries de **vazão, média diária** em toda a rede (campos: estado, estação, início e fim do registro).
2. `monitoring-locations`: por estado, só locais do tipo **Stream** (`site_type_code = ST`). Isso exclui canais, valas, lagos, nascentes e riachos de maré, que se comportam diferente de rio.

O período de registro dos metadados é só o primeiro e o último dia. **Ele não garante que não haja buracos no meio.** A completude real só aparece depois do download (seção 6).

In [16]:
ts_wanted = ["time_series_id", "monitoring_location_id", "state_name", "hydrologic_unit_code",
             "parameter_code", "statistic_id", "unit_of_measure", BEGIN_FIELD, END_FIELD,
             "computation_period_identifier", "primary", "sublocation_identifier", "web_description"]
ts_wanted = [c for c in ts_wanted if c in ts_props]

ts_params = {"parameter_code": PARAMETER_CODE, "statistic_id": STATISTIC_ID, "skipGeometry": "true"}
if not ALL_COLUMNS:
    ts_params["properties"] = ",".join(ts_wanted)
if "computation_period_identifier" in ts_props:
    ts_params["computation_period_identifier"] = "Daily"

def rename_period(df):
    # a versão antiga (begin/end, hora local) vira begin_local/end_local; a escolhida vira begin/end (UTC)
    legacy = {c: f"{c}_local" for c in ("begin", "end")
              if c in df.columns and c not in (BEGIN_FIELD, END_FIELD)}
    df = df.rename(columns=legacy).rename(columns={BEGIN_FIELD: "begin", END_FIELD: "end"})
    df["begin"] = pd.to_datetime(df["begin"], utc=True, errors="coerce")
    df["end"] = pd.to_datetime(df["end"], utc=True, errors="coerce")
    return df

cat_path = RAW_DIR / "catalog_timeseries.parquet"
if cat_path.exists():
    cat = pd.read_parquet(cat_path)
else:
    cat = fetch_items("time-series-metadata", ts_params)
    if cat.empty:
        raise RuntimeError("Catálogo vazio. Verifique PARAMETER_CODE/STATISTIC_ID e o valor de "
                           "'computation_period_identifier' (esperado: 'Daily').")
    if "time_series_id" not in cat or cat["time_series_id"].isna().all():
        cat["time_series_id"] = cat["_feature_id"]
    cat.to_parquet(cat_path)

cat = rename_period(cat).drop_duplicates("time_series_id")
print(f"{len(cat):,} séries de vazão diária no catálogo; {cat['monitoring_location_id'].nunique():,} estações; "
      f"{cat['state_name'].nunique()} estados/territórios")
print("Unidades:", cat["unit_of_measure"].value_counts(dropna=False).to_dict())
if "primary" in cat:
    print("Coluna 'primary':", cat["primary"].value_counts(dropna=False).to_dict())

25,909 séries de vazão diária no catálogo; 25,757 estações; 66 estados/territórios
Unidades: {'ft^3/s': 25909}
Coluna 'primary': {'Primary': 25849, nan: 60}


In [17]:
if STATES:
    cat = cat[cat["state_name"].isin(STATES)]

t0, t1 = pd.Timestamp(DATE_START, tz="UTC"), pd.Timestamp(DATE_END, tz="UTC")
tol = pd.Timedelta(days=1)   # tolerância: início/fim vêm em UTC e o dia civil é local
cat["overlaps"] = (cat["begin"] <= t1) & (cat["end"] >= t0)
cat["full_coverage"] = (cat["begin"] <= t0 + tol) & (cat["end"] >= t1 - tol)

ML_PROPS = ["id", "monitoring_location_name", "site_type_code", "site_type", "state_name",
            "county_name", "drainage_area", "hydrologic_unit_code", "agency_code"]

# ---- Controle de quais estados baixar ----
USE_TOP_STATE = True   # True = baixa só os estados candidatos a "mais estações"; False = todos
N_CANDIDATES = 3       # margem: o ranking vem do catálogo (antes do filtro ST), então olhamos os 3 primeiros

def fetch_stream_sites(state):
    """Locais ST de um estado, com cache em parquet. Estado sem locais gera um arquivo .empty
    para não ser consultado de novo."""
    tag = "all" if ALL_COLUMNS else "lean"
    stem = f"sites_ST_{state.replace(' ', '_')}_{tag}"
    path, empty_flag = RAW_DIR / f"{stem}.parquet", RAW_DIR / f"{stem}.empty"
    if path.exists():
        return pd.read_parquet(path)
    if empty_flag.exists():
        return pd.DataFrame()
    p = {"state_name": state, "site_type_code": "ST"}
    if not ALL_COLUMNS:
        p["properties"] = ",".join(ML_PROPS)
    df = fetch_items("monitoring-locations", p)
    if df.empty:
        empty_flag.touch()
    else:
        df.to_parquet(path)
    return df

# Estados elegíveis (mesma regra de cobertura da seção 4)
eligible = cat[cat["full_coverage"]] if REQUIRE_FULL_COVERAGE else cat[cat["overlaps"]]
ranking = eligible.groupby("state_name")["monitoring_location_id"].nunique().sort_values(ascending=False)

if STATES:
    states_to_fetch = sorted(cat["state_name"].dropna().unique())
elif USE_TOP_STATE:
    states_to_fetch = ranking.head(N_CANDIDATES).index.tolist()
else:
    states_to_fetch = sorted(cat["state_name"].dropna().unique())

print(f"Buscando locais do tipo Stream em {len(states_to_fetch)} estado(s): {states_to_fetch}")
sites = []
for s in states_to_fetch:
    df = fetch_stream_sites(s)
    print(f"  {s}: {len(df):,} locais ST", flush=True)
    if len(df):
        sites.append(df)

if not sites:
    raise RuntimeError("Nenhum local ST encontrado nos estados pedidos.")
sites = pd.concat(sites, ignore_index=True)
if "id" not in sites or sites["id"].isna().all():
    sites["id"] = sites["_feature_id"]
sites = sites.drop_duplicates("id").rename(columns={"id": "monitoring_location_id",
                                                    "state_name": "site_state_name",
                                                    "hydrologic_unit_code": "site_huc"})

cat_streams = cat.merge(sites, on="monitoring_location_id", how="inner")
print(f"\nSéries em locais Stream: {len(cat_streams):,} de {len(cat):,}")

Buscando locais do tipo Stream em 3 estado(s): ['California', 'Texas', 'Florida']
  California: 6,735 locais ST
  Texas: 3,079 locais ST
  Florida: 4,465 locais ST

Séries em locais Stream: 4,080 de 25,909


### 3.1 Estados disponíveis

In [18]:
state_summary = (cat_streams.groupby("state_name")
                 .agg(estacoes=("monitoring_location_id", "nunique"),
                      series=("time_series_id", "nunique"),
                      sobrepoem_2010_2025=("overlaps", "sum"),
                      cobrem_2010_2025_inteiro=("full_coverage", "sum"))
                 .sort_values("cobrem_2010_2025_inteiro", ascending=False))
state_summary.loc["TOTAL"] = state_summary.sum()
state_summary

,estacoes,series,sobrepoem_2010_2025,cobrem_2010_2025_inteiro
state_name,,,,
Texas,895,906,570,431
California,2359,2374,1005,402
Florida,798,800,389,273
TOTAL,4052,4080,1964,1106


### 3.2 Todas as estações

A tabela completa é grande e vai para CSV. Abaixo, uma amostra e as contagens.

In [10]:
station_cols = ["state_name", "monitoring_location_id", "monitoring_location_name", "county_name",
                "drainage_area", "site_huc", "lon", "lat", "begin", "end", "overlaps", "full_coverage",
                "time_series_id"]
station_cols = [c for c in station_cols if c in cat_streams.columns]
catalog_stations = cat_streams[station_cols].sort_values(["state_name", "monitoring_location_id"])
catalog_stations.to_csv(PROC_DIR / "catalog_stations.csv", index=False)
print(f"{len(catalog_stations):,} linhas salvas em {PROC_DIR / 'catalog_stations.csv'}")
catalog_stations.head(25)

4,080 linhas salvas em c:\Users\kayla\Meu Drive (kaylany.meneses18@gmail.com)\usgs_dados_streamflow\usgs_streamflow\data\usgs\processed\catalog_stations.csv


,state_name,monitoring_location_id,monitoring_location_name,county_name,drainage_area,site_huc,lon,lat,begin,end,overlaps,full_coverage,time_series_id
3524,California,USGS-09423350,CARUTHERS C NR IVANPAH CA,San Bernardino County,0.840,150301020203,-115.299,35.245,1963-10-01 07:00:00+00:00,2026-10-12 07:00:00+00:00,True,True,dd965c1f118b400fb9d2b0814585b31b
3557,California,USGS-09424050,CHEMEHUEVI WASH TRIB NR NEEDLES CA,San Bernardino County,2.040,150301010602,-114.604,34.508,1959-12-01 08:00:00+00:00,1968-09-29 07:00:00+00:00,False,False,df0b4675a393416da42866d322ef7f8b
1188,California,USGS-09424150,"COLORADO RIVER AQUEDUCT NEAR PARKER DAM, AZ-CA",San Bernardino County,NaN,150301010713,-114.157,34.316,1981-10-01 07:00:00+00:00,2022-09-29 07:00:00+00:00,True,False,497e78c0c1a84eecbfb4b173add9923a
3729,California,USGS-09424170,GENE WASH BL GENE WASH RES N PARKER DAM AZ-CAL,San Bernardino County,6.790,150301010713,-114.163,34.300,1965-10-01 07:00:00+00:00,1973-09-29 07:00:00+00:00,False,False,e8cbbc1451a24649937073076c2012c1
2608,California,USGS-09427520,"COLORADO RIVER BELOW PARKER DAM, AZ-CA",San Bernardino County,"182,700.000",150301040104,-114.140,34.296,1934-11-16 07:00:00+00:00,2022-09-05 06:00:00+00:00,True,False,a513bca0ac18457796d7c3b61b6f97e0
1216,California,USGS-09427600,DESILT WASH NEAR PARKER DAM-ARIZ.-CALIF,San Bernardino County,0.430,150301040104,-114.158,34.292,1965-10-01 07:00:00+00:00,1973-11-12 08:00:00+00:00,False,False,4b4ad0fdc0354affae096d12c8a259a0
3686,California,USGS-09427800,"EUREKA WASH NEAR PARKER DAM, AZ-CA",San Bernardino County,1.670,150301040104,-114.148,34.279,1965-10-01 07:00:00+00:00,1973-09-29 07:00:00+00:00,False,False,e6154b190a3641fdb0a793a58e9506a2
137,California,USGS-09428200,"COPPER BASIN WASH BELOW COPPER BASIN RESERVOIR,",San Bernardino County,11.600,150301040103,-114.198,34.254,1965-10-01 07:00:00+00:00,1973-11-12 08:00:00+00:00,False,False,09301b3a39ed4bd8afaa54cf771bb165
3875,California,USGS-09428530,ARCH C NR EARP CA,San Bernardino County,1.520,150301040304,-114.373,34.165,1960-10-01 08:00:00+00:00,1971-09-29 07:00:00+00:00,False,False,f2f45542203b44619dfce02a78da176d
116,California,USGS-09429010,"COLORADO RIVER AT PALO VERDE DAM, CA-AZ",Riverside County,"186,200.000",150301040406,-114.512,33.732,1969-04-01 08:00:00+00:00,1991-07-09 07:00:00+00:00,False,False,07bc550f565849c4abc194245db202ad


## 4. Seleção das estações

Regras: estações do tipo Stream, em `STATES` se definido, com cobertura 2010–2025 completa se `REQUIRE_FULL_COVERAGE`. Se uma estação tiver mais de uma série, fica a marcada como `primary` (se houver) e, no empate, a de início mais antigo. Com `MAX_SITES`, a amostra é estratificada por estado (round-robin), para não concentrar tudo em poucos estados.

In [11]:
def select_series(df, max_sites, seed):
    df = df.copy()
    multi = df.groupby("monitoring_location_id")["time_series_id"].nunique()
    print(f"Estações com mais de uma série candidata: {(multi > 1).sum():,}")
    df["_prim"] = df["primary"].astype(str).str.lower().eq("primary") if "primary" in df else False
    df = (df.sort_values(["monitoring_location_id", "_prim", "begin"], ascending=[True, False, True])
            .drop_duplicates("monitoring_location_id"))
    if max_sites is None or len(df) <= max_sites:
        return df
    rng = np.random.default_rng(seed)
    groups = {s: g.sample(frac=1, random_state=int(rng.integers(1_000_000_000)))
              for s, g in df.groupby("state_name")}
    picked, i = [], 0
    while len(picked) < max_sites and any(i < len(g) for g in groups.values()):
        for g in groups.values():
            if i < len(g) and len(picked) < max_sites:
                picked.append(g.iloc[i])
        i += 1
    return pd.DataFrame(picked)


pool = cat_streams[cat_streams["full_coverage"]] if REQUIRE_FULL_COVERAGE else cat_streams[cat_streams["overlaps"]]

counts = pool.groupby("state_name")["monitoring_location_id"].nunique().sort_values(ascending=False)
print("Estações por estado (top 5):")
print(counts.head(5).to_string())

if USE_TOP_STATE:
    TOP_STATE = counts.index[0]
    pool = pool[pool["state_name"] == TOP_STATE]
    print(f"\nEstado escolhido: {TOP_STATE} ({counts.iloc[0]} estações)")

selected = select_series(pool, MAX_SITES, SEED).reset_index(drop=True)
print(f"{len(selected)} estações selecionadas em {selected['state_name'].nunique()} estado(s)")
selected[["state_name", "monitoring_location_id", "monitoring_location_name", "drainage_area"]].head(20)

Estações por estado (top 5):
state_name
California    994
Texas         567
Florida       389

Estado escolhido: California (994 estações)
Estações com mais de uma série candidata: 5
994 estações selecionadas em 1 estado(s)


,state_name,monitoring_location_id,monitoring_location_name,drainage_area
0,California,USGS-09423350,CARUTHERS C NR IVANPAH CA,0.840
1,California,USGS-09424150,"COLORADO RIVER AQUEDUCT NEAR PARKER DAM, AZ-CA",NaN
2,California,USGS-09427520,"COLORADO RIVER BELOW PARKER DAM, AZ-CA","182,700.000"
3,California,USGS-09429100,"COLORADO RIVER BELOW PALO VERDE DAM, AZ-CA","182,200.000"
4,California,USGS-09429170,"PVID D-10-11-5 SPILL NR BLYTHE, CA",NaN
5,California,USGS-09429180,"PVID D23 SPILL NEAR BLYTHE, CA",NaN
6,California,USGS-09429190,"P.V.I.D. D-23-1 NEAR BLYTHE, CA",NaN
7,California,USGS-09429210,"P.V.I.D. C-28 UPPER SPILL NEAR BLYTHE, CA",NaN
8,California,USGS-09429230,"PVID C28 LOWER SPILL NEAR BLYTHE, CA.",NaN
9,California,USGS-09429490,"COLORADO RIVER ABOVE IMPERIAL DAM, AZ-CA","188,500.000"


## 5. Download da vazão diária

Lotes de `BATCH_SIZE` séries, período inteiro por lote, com cache em parquet. Cada lote devolve no máximo `8 x 5.479 = 43.832` linhas, abaixo do teto de 50.000. O código **confere** que o filtro de datas foi respeitado e que nenhuma linha veio de série não pedida. Se o servidor ignorar o filtro, ele para em vez de entregar dado errado.

In [12]:
DAILY_PROPS = [c for c in ["time_series_id", "monitoring_location_id", "time", "value",
                           "approval_status", "qualifier"] if c in daily_props]

def _clean_daily(df):
    if df.empty:
        return df
    for col in ("approval_status", "qualifier"):
        if col in df:
            df[col] = df[col].apply(lambda v: ";".join(map(str, v)) if isinstance(v, (list, tuple)) else v)
    df["time_api"] = df["time"]
    df["date"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None).dt.normalize()
    df["value_cfs"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.drop(columns=[c for c in ("time", "value", "_feature_id", "lon", "lat") if c in df])
    return df.drop_duplicates(["time_series_id", "date"], keep="last")

def fetch_daily_batch(ts_ids):
    key = hashlib.md5(",".join(sorted(ts_ids)).encode()).hexdigest()[:12]
    path = RAW_DIR / f"daily_{PARAMETER_CODE}_{STATISTIC_ID}_{DATE_START}_{DATE_END}_{key}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    params = {"time_series_id": ",".join(ts_ids), "datetime": f"{DATE_START}/{DATE_END}",
              "skipGeometry": "true"}
    if not ALL_COLUMNS:
        params["properties"] = ",".join(DAILY_PROPS)
    df = _clean_daily(fetch_items("daily", params))
    if len(df):
        stray = set(df["time_series_id"]) - set(ts_ids)
        if stray:
            raise RuntimeError(f"Linhas de séries não pedidas: {list(stray)[:3]}")
        if df["date"].min() < pd.Timestamp(DATE_START) or df["date"].max() > pd.Timestamp(DATE_END):
            raise RuntimeError("O servidor ignorou o filtro de datas (faixa devolvida fora de "
                               f"{DATE_START}..{DATE_END}). Revise o nome do parâmetro de tempo para esta versão.")
        if len(df) >= 50000:
            raise RuntimeError("Lote atingiu 50.000 linhas: possível truncamento. Reduza BATCH_SIZE.")
    df.to_parquet(path)
    return df

ids = selected["time_series_id"].tolist()
batches = [ids[i:i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)]
parts = []
for n, b in enumerate(batches, 1):
    print(f"lote {n}/{len(batches)} ({len(b)} séries)", flush=True)
    parts.append(fetch_daily_batch(b))
daily = pd.concat([p for p in parts if len(p)], ignore_index=True)
print(f"{len(daily):,} observações diárias de {daily['time_series_id'].nunique()} séries")
daily.head()

lote 1/125 (8 séries)
lote 2/125 (8 séries)
lote 3/125 (8 séries)
lote 4/125 (8 séries)
lote 5/125 (8 séries)
lote 6/125 (8 séries)
lote 7/125 (8 séries)
lote 8/125 (8 séries)
lote 9/125 (8 séries)
lote 10/125 (8 séries)
lote 11/125 (8 séries)
lote 12/125 (8 séries)
lote 13/125 (8 séries)
lote 14/125 (8 séries)
lote 15/125 (8 séries)
lote 16/125 (8 séries)
lote 17/125 (8 séries)
lote 18/125 (8 séries)
lote 19/125 (8 séries)
lote 20/125 (8 séries)
lote 21/125 (8 séries)
lote 22/125 (8 séries)
lote 23/125 (8 séries)
lote 24/125 (8 séries)
lote 25/125 (8 séries)
lote 26/125 (8 séries)
lote 27/125 (8 séries)
lote 28/125 (8 séries)
lote 29/125 (8 séries)
lote 30/125 (8 séries)
lote 31/125 (8 séries)
lote 32/125 (8 séries)
lote 33/125 (8 séries)
lote 34/125 (8 séries)
lote 35/125 (8 séries)
lote 36/125 (8 séries)
lote 37/125 (8 séries)
lote 38/125 (8 séries)
lote 39/125 (8 séries)
lote 40/125 (8 séries)
lote 41/125 (8 séries)
lote 42/125 (8 séries)
lote 43/125 (8 séries)
lote 44/125 (8 série

,time_series_id,monitoring_location_id,parameter_code,statistic_id,unit_of_measure,approval_status,qualifier,last_modified,time_api,date,value_cfs
0,497e78c0c1a84eecbfb4b173add9923a,USGS-09424150,00060,00003,ft^3/s,Approved,NaN,2025-07-05T00:14:52.573320+00:00,2014-06-22,2014-06-22,"1,690.000"
1,03e4a517b090406a9198a40a4fe228c2,USGS-09429100,00060,00003,ft^3/s,Approved,NaN,2025-03-10T19:58:36.324383+00:00,2022-05-19,2022-05-19,"9,530.000"
2,360e1e9b9b634d47af3e3a98204ecd45,USGS-09429180,00060,00003,ft^3/s,Approved,NaN,2025-03-10T20:04:43.121116+00:00,2011-07-21,2011-07-21,24.300
3,497e78c0c1a84eecbfb4b173add9923a,USGS-09424150,00060,00003,ft^3/s,Approved,NaN,2025-07-05T00:14:52.573320+00:00,2018-02-21,2018-02-21,0.000
4,dd965c1f118b400fb9d2b0814585b31b,USGS-09423350,00060,00003,ft^3/s,Approved,ESTIMATED,2025-03-10T23:02:56.154357+00:00,2011-01-07,2011-01-07,0.000


## 6.2 DataFrame final: grade diária completa e dicionário de dados

`df_full` contém uma linha para cada combinação de `time_series_id` selecionado e cada dia entre `DATE_START` e `DATE_END`. A grade é combinada com as observações da USGS e com os metadados do local. Não há imputação nesta etapa: datas sem valor continuam como `NaN`.

### Como a API organiza os dados

A API OGC da USGS separa três entidades principais usadas neste notebook:

- [`monitoring-locations`](https://api.waterdata.usgs.gov/ogcapi/v1/collections/monitoring-locations/schema?f=json): local físico monitorado, com identificador, coordenadas e características do local.
- [`time-series-metadata`](https://api.waterdata.usgs.gov/ogcapi/v1/collections/time-series-metadata/schema?f=json): série de um parâmetro em um local, com unidade, estatística e período de registro. O início/fim não garante ausência de lacunas.
- [`daily`](https://api.waterdata.usgs.gov/ogcapi/v1/collections/daily/schema?f=json): observações diárias com tempo, valor, unidade, aprovação e qualificadores.

`monitoring_location_id` identifica o local físico; `time_series_id` identifica a série e é usado para relacionar observações e metadados. Neste notebook, `parameter_code = 00060` seleciona vazão e `statistic_id = 00003` seleciona média diária. Consulte o [portal da API](https://api.waterdata.usgs.gov/ogcapi/v1?f=html), os [códigos de parâmetros](https://api.waterdata.usgs.gov/ogcapi/v1/collections/parameter-codes/items?f=html) e os [códigos de estatísticas](https://api.waterdata.usgs.gov/ogcapi/v1/collections/statistic-codes/items?f=html). Os esquemas oficiais são a referência para nomes e descrições dos campos; confirme-os para a versão definida em `API_VERSION`.

### Colunas de `df_full`

| Coluna | Significado |
|---|---|
| `date` | Data diária da grade. Confira a interpretação de data/fuso ao alinhar fontes distintas. |
| `monitoring_location_id` | Identificador USGS do local físico. |
| `time_series_id` | Identificador da série USGS selecionada e chave de junção das observações. |
| `monitoring_location_name` | Nome descritivo do local. |
| `state_name`, `county_name` | Estado/território e condado informados pela API. |
| `lat`, `lon` | Coordenadas do local, quando disponíveis. |
| `drainage_area` | Área de drenagem do metadado; confirme a unidade antes de conversões ou do balanço hídrico. |
| `altitude` | Altitude do local; verifique unidade e datum antes de uso quantitativo. |
| `site_huc` | Código HUC associado ao local. |
| `value_cfs` | Vazão diária na unidade original; para `00060`, ft³/s. |
| `value_m3s` | Vazão convertida de ft³/s para m³/s; `NaN` permanece `NaN`. |
| `unit_of_measure` | Unidade declarada pela USGS para a série. |
| `approval_status` | Status como `Approved` ou `Provisional`; dados provisórios podem ser revisados. |
| `qualifier` | Qualificadores da observação, como estimativa ou possível influência de gelo, quando informados. |
| `record_returned` | `True` se a API retornou uma linha para a série/data; `False` se não houve linha correspondente. |
| `is_missing` | `True` quando `value_cfs` não tem valor numérico utilizável; inclui lacunas da grade e registros sem valor numérico. |

**Zero não é ausência:** `value_cfs == 0` é uma medição válida. `record_returned=False` indica data sem registro retornado; `record_returned=True` e `is_missing=True` indica registro retornado sem valor numérico utilizável. Não imputar nem remover esses `NaN` antes de avaliar cobertura e criar máscaras de observação.

### Considerações para o objetivo do projeto

O objetivo em estudo é reconstruir trechos ausentes de vazão e comparar modelos com e sem informação hidrometeorológica/restrições físicas. Para avaliar a reconstrução, oculte trechos conhecidos durante a validação e compare as estimativas com os valores originais.

A coleção [`continuous`](https://api.waterdata.usgs.gov/ogcapi/v1/collections/continuous/schema?f=json) contém dados de alta frequência, muitas vezes em intervalos de 15 minutos, mas frequência e cobertura dependem da estação e do parâmetro. Vazão (`00060`), nível (`00065`) e precipitação (`00045`) podem ser úteis onde disponíveis. Chuva medida num local é pontual, não necessariamente a média da bacia; evapotranspiração e forçantes espaciais podem exigir fonte externa. Não misture séries sub-horárias com forçantes diárias sem estratégia temporal explícita.

Veja também o [README do projeto](../README.md) para escopo, execução, arquivos gerados e referências adicionais.

In [ ]:
import gc
import pandas as pd

# IDs das séries selecionadas e calendário completo do período
series_ids = selected["time_series_id"].dropna().drop_duplicates().tolist()
calendar = pd.date_range(DATE_START, DATE_END, freq="D", name="date")

#Uma linha para cada série em cada dia, mesmo que não exista observação
grid = pd.MultiIndex.from_product(
    [series_ids, calendar],
    names=["time_series_id", "date"],
).to_frame(index=False)

# Mapeia cada série para sua estação
ts_to_site = selected.set_index("time_series_id")["monitoring_location_id"]

#Mantém as observações retornadas pela USGS
obs_cols = [
    "time_series_id",
    "date",
    "value_cfs",
    "approval_status",
    "qualifier",
]
obs_cols = [col for col in obs_cols if col in daily.columns]

obs = daily.loc[
    daily["time_series_id"].isin(series_ids),
    obs_cols,
].copy()

obs["date"] = pd.to_datetime(obs["date"]).dt.normalize()

#Evita mascarar duplicatas: cada série deve ter no máximo uma linha por dia
if obs.duplicated(["time_series_id", "date"]).any():
    raise ValueError("Há observações duplicadas para uma série e uma data.")

#Marca que a linha veio da API; isso diferencia ausência de registro de valor zero
obs["record_returned"] = True

#Completa a grade: dias sem linha retornada pela API ficam com NaN
obs = grid.merge(
    obs,
    on=["time_series_id", "date"],
    how="left",
    validate="one_to_one",
)

obs["record_returned"] = obs["record_returned"].fillna(False).astype(bool)
obs["monitoring_location_id"] = obs["time_series_id"].map(ts_to_site)

#Converte a vazão; NaN permanece NaN e zero continua sendo uma medição válida
obs["value_m3s"] = obs["value_cfs"] * CFS_TO_M3S
obs["is_missing"] = obs["value_cfs"].isna()

#Metadados de todas as séries selecionadas, inclusive as sem observações
stations_full = selected.copy()
stations_full = stations_full.rename(
    columns={"begin": "registro_inicio", "end": "registro_fim"}
)
stations_full = stations_full.drop(
    columns=[
        col for col in stations_full.columns
        if col.startswith("_feature_id")
        or col in ("_prim", "overlaps", "full_coverage")
    ]
)

meta_cols = [
    "time_series_id",
    "monitoring_location_name",
    "state_name",
    "county_name",
    "lat",
    "lon",
    "drainage_area",
    "altitude",
    "site_huc",
    "unit_of_measure",
]
meta = stations_full[[col for col in meta_cols if col in stations_full.columns]].copy()

# Acrescenta os metadados a cada série/dia
df_full = obs.merge(
    meta,
    on="time_series_id",
    how="left",
    validate="many_to_one",
)

# Usa categorias para reduzir o consumo de memória
for col in (
    "time_series_id",
    "monitoring_location_id",
    "monitoring_location_name",
    "state_name",
    "county_name",
    "unit_of_measure",
    "approval_status",
    "qualifier",
    "site_huc",
):
    if col in df_full.columns:
        df_full[col] = df_full[col].astype("category")

# Organiza as colunas e as linhas
first_cols = [
    "date",
    "monitoring_location_id",
    "monitoring_location_name",
    "state_name",
    "county_name",
    "lat",
    "lon",
    "value_cfs",
    "value_m3s",
    "unit_of_measure",
    "approval_status",
    "qualifier",
    "record_returned",
    "is_missing",
]
other_cols = [col for col in df_full.columns if col not in first_cols]
df_full = df_full[
    [col for col in first_cols if col in df_full.columns] + other_cols
]
df_full = df_full.sort_values(
    ["monitoring_location_id", "date"]
).reset_index(drop=True)

print(
    f"{len(df_full):,} linhas x {len(df_full.columns)} colunas | "
    f"{df_full['monitoring_location_id'].nunique()} estações"
)
print(f"Dias sem valor: {df_full['is_missing'].sum():,}")
print(f"Memória do df_full: {df_full.memory_usage(deep=True).sum() / 1e9:.2f} GB")

df_full.head(10)

5,446,126 linhas x 18 colunas | 994 estações
Dias sem valor: 1,136,828
Memória do df_full: 0.39 GB


,date,monitoring_location_id,monitoring_location_name,state_name,county_name,lat,lon,value_cfs,value_m3s,unit_of_measure,approval_status,qualifier,record_returned,is_missing,time_series_id,drainage_area,altitude,site_huc
0,2010-01-01,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
1,2010-01-02,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
2,2010-01-03,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
3,2010-01-04,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
4,2010-01-05,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
5,2010-01-06,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
6,2010-01-07,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
7,2010-01-08,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
8,2010-01-09,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203
9,2010-01-10,USGS-09423350,CARUTHERS C NR IVANPAH CA,California,San Bernardino County,35.245,-115.299,0.000,0.000,ft^3/s,Approved,NaN,True,False,dd965c1f118b400fb9d2b0814585b31b,0.840,"5,656.230",150301020203


In [14]:
df_full.columns

Index(['date', 'monitoring_location_id', 'monitoring_location_name',
       'state_name', 'county_name', 'lat', 'lon', 'value_cfs', 'value_m3s',
       'unit_of_measure', 'approval_status', 'qualifier', 'record_returned',
       'is_missing', 'time_series_id', 'drainage_area', 'altitude',
       'site_huc'],
      dtype='str')